# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and initialize mlcroissant Dataset object
dataset = mlc.Dataset(url)

# Display dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available RecordSets and fields. All entities are referenced by their `@id` values.

Let's inspect all available RecordSets, their IDs, and their fields.

In [ ]:
# List all RecordSets and their fields, referenced by `@id`
record_sets = list(dataset.record_sets)
print(f"Available RecordSets (by @id):\n------------------------------")
for rs in record_sets:
    print(f'RecordSet @id: {rs.id} | name: {rs.name}')
    print('  Fields:')
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")
    print('  Columns:')
    for col in rs.columns:
        print(f"    - {col.id} (name: {col.name})")
    print('')

## 3. Data Extraction
Load data from a specific RecordSet into a DataFrame for analysis.

We select the main protein abundance record set for demonstration. All references use their `@id`.

In [ ]:
# Pick the main RecordSet for protein abundance table
# (Replace string below with the printed @id for protein abundance table. Example shown is an educated guess.)
main_record_set_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/recordset/protein_abundance_table'

# Assemble a list of all record set @ids
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, shape: {dataframes[record_set_id].shape}")

columns = dataframes[main_record_set_id].columns.tolist()
print(f"\nColumns in main protein abundance table (@id: {main_record_set_id}):")
print(columns)
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing a field, and grouping.

In [ ]:
# For demonstration, pick an actual numeric field by examining the printed columns
# Let's assume the field @id for abundance is 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/field/abundance_sample_1'
numeric_field = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/field/abundance_sample_1'  # Replace with actual @id from printed columns
# For grouping, use protein description or accession, e.g.:
group_field = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/field/description'  # Replace with actual @id as needed

# Remove missing values for the numeric field
main_df = dataframes[main_record_set_id].copy()
main_df = main_df[main_df[numeric_field].notnull()]

# Assume field is numeric; if not, coerce/convert
main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
main_df = main_df[main_df[numeric_field].notnull()]

# Filter by threshold on abundance
threshold = 10.0
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with field {numeric_field} > {threshold}:")
print(filtered_df[[numeric_field]].head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

# Group by protein description, if present
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution and its normalization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the original abundance field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f'Distribution of Abundance Field: {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Plot histogram of the normalized field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True)
plt.title(f'Normalized Abundance Field: {numeric_field}')
plt.xlabel(f'{numeric_field}_normalized')
plt.ylabel('Count')
plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 protein abundance dataset using the `mlcroissant` library. Using only entity `@id` references, we loaded multiple record sets, selected fields for analysis, filtered, normalized, and visualized numerical protein abundance data. This structured approach enables transparent, reproducible analysis and downstream biological insights.

**Key findings:**
- The dataset provides detailed protein abundance with rich annotation for each field and record set via unique `@id`s.
- The main protein table allows numerical EDA, filtering and normalization.
- Using only `@id` ensures stability and schema-agnostic processing with Croissant datasets.

You can extend this notebook by experimenting with other record sets, fields, or advanced analyses.